In [3]:
import websocket
import json
import numpy
import talib
import config
import pprint
from binance.enums import *
from binance.client import Client

ModuleNotFoundError: No module named 'talib'

In [ ]:
RSI_PERIOD = 14
OVERSOLD_THRESHOLD = 40
OVERBOUGHT_THRESHOLD = 60
TRADE_QUANTITY = 0.005
TRADE_SYMBOL = 'BTCUSD'

closes = []
position = False

SOCKET = 'wss://stream.binance.com:9443/ws/btcusdt@kline_1m'

def on_message(ws, message):
    print("message received")
    json_message = json.loads(message)
    #pprint.pprint(json_message)
    candle = json_message['k']
    is_candle_closed = candle['x']
    close = candle['c']
    if is_candle_closed:
        closes.append(float(close))
        print("candle closed at {}".format(close))
        
        if len(closes) > RSI_PERIOD:
            np_closes = numpy.array(closes)
            rsi = talib.RSI(np_closes,  RSI_PERIOD)
            print("all rsis calculated so far")
            print(rsi)
            last_rsi = rsi[-1]
            print("current rs is" + last_rsi)

            if last_rsi > OVERBOUGHT_THRESHOLD and not position:
                print("Buy btc")
            if last_rsi < OVERSOLD_THRESHOLD and position:
                print("Sell btc")
                    
                    
def on_error(ws, error):
    print(error)

def on_close(ws, close_status_code, close_msg):
    print("### closed ###")

def on_open(ws):
    print("### opened ###")

# websocket.enableTrace(True)
ws = websocket.WebSocketApp(SOCKET, on_open=on_open, on_message=on_message, on_error=on_error, on_close=on_close)
ws.run_forever()

In [ ]:
closes = []
client = Client(config.API_KEY, confic.API_SECRET, tld='us ')